
**content-based recommender system**

A content-based recommender system suggests items similar to what a user has liked in the past. It uses item features such as genre, description, tags, cast, or keywords to compute similarity and recommend relevant items.


**Advantages:**

*   Personalized recommendations

* No dependency on other users

* Works well when user preferences are consistent



**Limitations:**


*  Limited discovery (over-specialization)

* Requires good quality item metadata

* Cold-start problem for new items



In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
movies = pd.read_csv("/content/movies.csv")
ratings = pd.read_csv("/content/ratings.csv")
tags = pd.read_csv("/content/tags.csv")

movies.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
movie_ratings = ratings.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()


In [ ]:
popular_movies = pd.merge(movies, movie_ratings, on="movieId")


In [ ]:
popular_movies = popular_movies[popular_movies["rating_count"] >= 50]
popular_movies = popular_movies.sort_values(
    by=["avg_rating", "rating_count"], ascending=False
)

popular_movies.head(10)[["title", "avg_rating", "rating_count"]]


,title,avg_rating,rating_count
277,"Shawshank Redemption, The (1994)",4.429022,317
659,"Godfather, The (1972)",4.289062,192
2224,Fight Club (1999),4.272936,218
974,Cool Hand Luke (1967),4.271930,57
602,Dr. Strangelove or: How I Learned to Stop Worr...,4.268041,97
686,Rear Window (1954),4.261905,84
921,"Godfather: Part II, The (1974)",4.259690,129
6298,"Departed, The (2006)",4.252336,107
913,Goodfellas (1990),4.250000,126
694,Casablanca (1942),4.240000,100


In [ ]:
tags_grouped = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x)).reset_index()


In [ ]:
movies_cb = pd.merge(movies, tags_grouped, on="movieId", how="left")
movies_cb["tag"] = movies_cb["tag"].fillna("")


In [ ]:
movies_cb["content"] = movies_cb["genres"] + " " + movies_cb["tag"]
movies_cb.head()


,movieId,title,genres,tag,content
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun,Adventure|Animation|Children|Comedy|Fantasy pi...
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game,Adventure|Children|Fantasy fantasy magic board...
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old,Comedy|Romance moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,,Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake,Comedy pregnancy remake


In [ ]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies_cb["content"])


In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


In [ ]:
def recommend_movies(movie_title, num_recommendations=5):
    idx = movies_cb[movies_cb["title"] == movie_title].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:num_recommendations+1]
    movie_indices = [i[0] for i in sim_scores]

    return movies_cb.iloc[movie_indices][["title", "genres"]]


In [ ]:
recommend_movies("Toy Story (1995)", 5)


,title,genres
1757,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
8695,Guardians of the Galaxy 2 (2017),Action|Adventure|Sci-Fi
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy


This project successfully implemented popularity-based and content-based movie recommender systems using the MovieLens dataset. Popularity-based recommendations highlight widely liked movies, while content-based filtering provides personalized suggestions using genres and tags. The project demonstrates how Python and machine learning techniques can be effectively used to build practical recommendation systems.